In [1]:
# 1. 导入库
# 2.超参数设置 
# 3. 定义模型（CNN）
# 4. 优化函数、损失函数定义 
# 5. dataset\dataloader
# 6. 训练函数
# 7. 测试函数
# 8. 运行训练与评估

In [2]:
!pip install tqdm


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import torch
from torch import nn,optim
from torch.nn import functional as F

#导入torchvision读取图片
from torchvision import datasets,transforms
#MNIST可通过pytorchapi直接接入
from torchvision.datasets import MNIST

#导入dataloader加载数据
from torch.utils.data import DataLoader

#可视化
from tqdm import tqdm

In [8]:
#超参数
device="cuda:0"
batch_size=256
lr=1e-4
epochs=15


In [9]:
#模型定义：
class Lenet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv=nn.Sequential(
            #conv0
            nn.Conv2d(1,6,kernel_size=5,bias=True),
        #BatchNorm层在批归一化
            nn.BatchNorm2d(6),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            #conv1
            nn.Conv2d(6,16,kernel_size=5,bias=True),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

         
        )
        self.fc=nn.Sequential(
            nn.Linear(16*5*5,120),
            nn.ReLU(),
            nn.Linear(120,84),
            nn.ReLU(),
            nn.Linear(84,10),
        )
    
            
    
    def forward(self,x):
        x=self.conv(x)
        x=torch.flatten(x,start_dim=1)
        x=self.fc(x)
        return x
            
model=Lenet()    
model.to(device) #记得迁移到gpu上去
        

Lenet(
  (conv): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (5): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=400, out_features=120, bias=True)
    (1): ReLU()
    (2): Linear(in_features=120, out_features=84, bias=True)
    (3): ReLU()
    (4): Linear(in_features=84, out_features=10, bias=True)
  )
)

In [15]:
#优化函数定义
criterion=nn.CrossEntropyLoss()
optimizer=optim.AdamW(model.parameters(),lr=lr) #必须要在模型之后

## 目前最不熟悉的板块：数据加载

In [11]:
#数据加载：数据预处理
transform = transforms.Compose([
    transforms.Pad(2),#默认28x28的尺寸填充为32x32尺寸
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.5],std = [0.5])

    ]) #预处理函数，在下载时作为参数
path='/data/'

#只是下载，返回dataset对象
train_dataset=MNIST(path,train=True,transform=transform,download=True)  #下载，并执行预处理
test_dataset=MNIST(path,train=False,transform=transform)

#分割为一份一份batch,训练时的batch_x、batch_y来源
train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)#shuffle是为了打乱顺序
test_loader=DataLoader(test_dataset,batch_size=64,shuffle=False)



#如果自定义dataloader，目标是返回（x-y）的tuple
'''
class MyDataLoader:
    def __init__(self, dataset, batch_size, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = list(range(len(dataset)))
    
    def __iter__(self):
        if self.shuffle:
            random.shuffle(self.indices)
        self.current = 0
        return self
    
    def __next__(self):
        if self.current >= len(self.indices):
            raise StopIteration
        batch_indices = self.indices[self.current:self.current + self.batch_size]
        batch = [self.dataset[i] for i in batch_indices]
        # 假设 dataset[i] 返回 (x, y)
        xs = torch.stack([item[0] for item in batch])
        ys = torch.tensor([item[1] for item in batch])
        self.current += self.batch_size
        return xs, ys
'''

'\nclass MyDataLoader:\n    def __init__(self, dataset, batch_size, shuffle=True):\n        self.dataset = dataset\n        self.batch_size = batch_size\n        self.shuffle = shuffle\n        self.indices = list(range(len(dataset)))\n\n    def __iter__(self):\n        if self.shuffle:\n            random.shuffle(self.indices)\n        self.current = 0\n        return self\n\n    def __next__(self):\n        if self.current >= len(self.indices):\n            raise StopIteration\n        batch_indices = self.indices[self.current:self.current + self.batch_size]\n        batch = [self.dataset[i] for i in batch_indices]\n        # 假设 dataset[i] 返回 (x, y)\n        xs = torch.stack([item[0] for item in batch])\n        ys = torch.tensor([item[1] for item in batch])\n        self.current += self.batch_size\n        return xs, ys\n'

In [18]:
#训练函数
for epoch in range(epochs):
    epoch_loss=0
    for batch_x,batch_y in tqdm(train_loader):
        #一定要保证模型与数据都迁移了
        batch_x = batch_x.to(device)  
        batch_y = batch_y.to(device)
        y_pred=model(batch_x)
        loss=criterion(y_pred,batch_y)
        epoch_loss+=loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    if epoch%10==0:
        print(f'epoch:{epoch},loss:{epoch_loss/batch_size}')


100%|██████████| 938/938 [00:18<00:00, 49.86it/s]


epoch:0,loss:0.026885641738772392


100%|██████████| 938/938 [00:20<00:00, 46.55it/s]


epoch:10,loss:0.015158923342823982


100%|██████████| 938/938 [00:19<00:00, 48.13it/s]


# 最不熟悉板块：检验（含权重存储、加载）模块

In [19]:
torch.save(model.state_dict(),'lenet_mnist.params')



# 1. 加载模型
test_model = Lenet()
test_model.load_state_dict(torch.load('lenet_mnist.params',weights_only=False))
test_model.to(device)  # 模型移到 GPU
test_model.eval()       # 切换到评估模式

# 2. 测试
correct = 0
total = 0

with torch.no_grad():  # 测试时不需要计算梯度     train的外侧是for循环，test则是关闭梯度计算：内部几乎一致
    for batch_x, batch_y in tqdm(test_loader):
        #几乎与train部分一致，保证都迁移到gpu
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        output = test_model(batch_x)#output是一个单维度向量，train的时候的y_pred

        # 取概率最大的类别作为预测结果
        _, predicted = torch.max(output, 1)

        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = 100 * correct / total
print(f'测试集准确率: {accuracy:.2f}%')

100%|██████████| 157/157 [00:03<00:00, 49.47it/s]

测试集准确率: 99.09%
